In [ ]:
import pandas as pd
from pathlib import Path

PROCESSED_DIR = Path(r"C:\Projet_filrouge\oncobio_decision_analytics\01_Data\Processed")

patients_master = pd.read_csv(PROCESSED_DIR / "patients_master.csv")
biomarkers = pd.read_csv(PROCESSED_DIR / "biomarker_reference_clean.csv")
survival_summary = pd.read_csv(PROCESSED_DIR / "survival_summary.csv")

patients_master.head()

### 2. KPI cliniques

In [ ]:
# Total patients
total_patients = len(patients_master)

# Age moyen
mean_age = round(patients_master["age"].mean(), 1)

# Répartition par sexe
sex_distribution = (
    patients_master["sex"]
    .value_counts(dropna=False)
    .reset_index()
)
sex_distribution.columns = ["sex", "count"]
sex_distribution["percentage"] = round(sex_distribution["count"] / total_patients * 100, 2)

# Répartition par type de cancer
cancer_distribution = (
    patients_master["cancer_type"]
    .value_counts()
    .reset_index()
)
cancer_distribution.columns = ["cancer_type", "count"]
cancer_distribution["percentage"] = round(cancer_distribution["count"] / total_patients * 100, 2)

# Répartition par groupe de risque
risk_distribution = (
    patients_master["risk_group"]
    .value_counts(dropna=False)
    .reset_index()
)
risk_distribution.columns = ["risk_group", "count"]
risk_distribution["percentage"] = round(risk_distribution["count"] / total_patients * 100, 2)

print("Total patients:", total_patients)
print("Mean age:", mean_age)
display(sex_distribution)
display(cancer_distribution)
display(risk_distribution)

### 3. KPI pronostiques

In [ ]:
# Taux de décès global
death_rate_global = round(patients_master["event"].mean() * 100, 2)

# Médiane de survie
median_survival_global = round(patients_master["survival_months"].median(), 1)

# Survie moyenne
mean_survival_global = round(patients_master["survival_months"].mean(), 1)

# Survie par stade
survival_by_stage = (
    patients_master.groupby("stage", dropna=False)
    .agg(
        total_patients=("patient_id", "count"),
        death_rate=("event", "mean"),
        mean_survival=("survival_months", "mean"),
        median_survival=("survival_months", "median")
    )
    .reset_index()
)

survival_by_stage["death_rate"] = round(survival_by_stage["death_rate"] * 100, 2)
survival_by_stage["mean_survival"] = round(survival_by_stage["mean_survival"], 1)
survival_by_stage["median_survival"] = round(survival_by_stage["median_survival"], 1)

# Survie par ECOG
survival_by_ecog = (
    patients_master.groupby("ecog", dropna=False)
    .agg(
        total_patients=("patient_id", "count"),
        death_rate=("event", "mean"),
        mean_survival=("survival_months", "mean"),
        median_survival=("survival_months", "median")
    )
    .reset_index()
)

survival_by_ecog["death_rate"] = round(survival_by_ecog["death_rate"] * 100, 2)
survival_by_ecog["mean_survival"] = round(survival_by_ecog["mean_survival"], 1)
survival_by_ecog["median_survival"] = round(survival_by_ecog["median_survival"], 1)

print("Global death rate:", death_rate_global, "%")
print("Global median survival:", median_survival_global)
print("Global mean survival:", mean_survival_global)

display(survival_by_stage)
display(survival_by_ecog)

### 4. KPI décisionnels

In [ ]:
# Biomarqueurs les plus étudiés
top_biomarkers = (
    biomarkers["gene_symbol"]
    .value_counts()
    .reset_index()
)
top_biomarkers.columns = ["gene_symbol", "count"]

# Groupes les plus à risque
high_risk_profiles = (
    patients_master[patients_master["risk_group"] == "High Risk"]
    .groupby(["cancer_type", "sex"])
    .agg(
        total_patients=("patient_id", "count"),
        mean_age=("age", "mean"),
        death_rate=("event", "mean")
    )
    .reset_index()
)

high_risk_profiles["mean_age"] = round(high_risk_profiles["mean_age"], 1)
high_risk_profiles["death_rate"] = round(high_risk_profiles["death_rate"] * 100, 2)

# Profils faible survie
low_survival_profiles = (
    patients_master.groupby(["cancer_type", "risk_group"])
    .agg(
        total_patients=("patient_id", "count"),
        median_survival=("survival_months", "median"),
        death_rate=("event", "mean")
    )
    .reset_index()
)

low_survival_profiles["death_rate"] = round(low_survival_profiles["death_rate"] * 100, 2)
low_survival_profiles["median_survival"] = round(low_survival_profiles["median_survival"], 1)

display(top_biomarkers.head(10))
display(high_risk_profiles)
display(low_survival_profiles)

### 5. Export des KPI finaux

In [ ]:
sex_distribution.to_csv(PROCESSED_DIR / "kpi_sex_distribution.csv", index=False)
cancer_distribution.to_csv(PROCESSED_DIR / "kpi_cancer_distribution.csv", index=False)
risk_distribution.to_csv(PROCESSED_DIR / "kpi_risk_distribution.csv", index=False)
survival_by_stage.to_csv(PROCESSED_DIR / "kpi_survival_by_stage.csv", index=False)
survival_by_ecog.to_csv(PROCESSED_DIR / "kpi_survival_by_ecog.csv", index=False)
high_risk_profiles.to_csv(PROCESSED_DIR / "kpi_high_risk_profiles.csv", index=False)
low_survival_profiles.to_csv(PROCESSED_DIR / "kpi_low_survival_profiles.csv", index=False)

print("KPI exports terminés.")